In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
import random
import warnings
warnings.filterwarnings("ignore")

StatementMeta(, 1cd907cf-3722-4500-b222-5e782a969438, 3, Finished, Available, Finished, False)

In [2]:
np.random.seed(42)
random.seed(42)

StatementMeta(, 1cd907cf-3722-4500-b222-5e782a969438, 4, Finished, Available, Finished, False)

In [3]:

CONFIG = {
    "start_date": "2025-01-01",
    "end_date": str(date.today()),
    "n_products": 30,
    "n_warehouses": 3,
    "n_transactions": 800,
    "lakehouse_name": "inventory_lakehouse",
}

print("✅ Config loaded:", CONFIG)
print(f"📅 Period: {CONFIG['start_date']} → {CONFIG['end_date']}")
print(f"📦 Products: {CONFIG['n_products']} | Transactions: {CONFIG['n_transactions']}")

StatementMeta(, 1cd907cf-3722-4500-b222-5e782a969438, 5, Finished, Available, Finished, False)

✅ Config loaded: {'start_date': '2025-01-01', 'end_date': '2026-03-11', 'n_products': 30, 'n_warehouses': 3, 'n_transactions': 800, 'lakehouse_name': 'inventory_lakehouse'}
📅 Period: 2025-01-01 → 2026-03-11
📦 Products: 30 | Transactions: 800


In [4]:
# ============================================================
# CELL 2: Dimension Tables
# ============================================================

# ── DIM_PRODUCT ─────────────────────────────────────────────
categories = {
    "Electronics": [
        'LED Monitor 24"',
        "Mechanical Keyboard",
        "Gaming Mouse",
        "Bluetooth Headset",
        "HD Webcam",
        "USB-C Hub",
    ],
    "Office Supplies": [
        "A4 Copy Paper (500s)",
        "HP Ink Cartridge",
        "Ballpoint Pens (10-pack)",
        "Document Binder",
        "Adhesive Tape",
        "Padded Envelopes",
    ],
    "Components": [
        "16GB DDR4 RAM",
        "512GB SSD",
        "Intel Core i5 CPU",
        "ATX Motherboard",
        "Graphics Card RTX 3060",
        "650W PSU",
    ],
    "Accessories": [
        'HDMI Cable 2m',
        "Laptop Stand",
        "USB-C Charging Cable",
        "Screen Protector",
        'Laptop Sleeve 15"',
        "Cooling Pad",
    ],
    "Consumables": [
        "AA Batteries (8-pack)",
        "Lens Cleaning Wipes",
        "Screen Cleaning Spray",
        "Thermal Paste",
        "Isopropyl Alcohol 90%",
        "Zip-lock Bags (100s)",
    ],
}

base_prices_usd = {
    "Electronics": (50, 600),
    "Office Supplies": (3, 80),
    "Components": (40, 900),
    "Accessories": (8, 120),
    "Consumables": (3, 40),
}

suppliers = [
    "TechSupply Co.",
    "KeyCraft Ltd.",
    "MemCore EU",
    "ConnectPro SA",
]

products = []
pid = 1

for cat, items in categories.items():
    lo, hi = base_prices_usd[cat]
    
    for name in items:
        cost = np.random.randint(lo, hi)
        margin = np.random.uniform(0.15, 0.45)
        
        products.append({
            "product_id": pid,
            "product_name": name,
            "category": cat,
            "unit": "pcs",
            "currency": "USD",
            "base_cost": cost,
            "base_price": int(cost * (1 + margin)),
            "reorder_point": np.random.randint(5, 20),
            "safety_stock": np.random.randint(3, 10),
            "lead_time_days": np.random.choice([3, 5, 7, 14]),
            "supplier": np.random.choice(suppliers),
            "is_active": 1,
        })
        
        pid += 1

dim_product = pd.DataFrame(products)

# ── DIM_WAREHOUSE ─────────────────────────────────────────────
dim_warehouse = pd.DataFrame([
    {
        "warehouse_id": 1,
        "warehouse_name": "NYC Warehouse",
        "city": "New York",
        "country": "United States",
        "region": "North America",
        "capacity_sqm": 500,
        "timezone": "America/New_York",
    },
    {
        "warehouse_id": 2,
        "warehouse_name": "LA Warehouse",
        "city": "Los Angeles",
        "country": "United States",
        "region": "North America",
        "capacity_sqm": 800,
        "timezone": "America/Los_Angeles",
    },
    {
        "warehouse_id": 3,
        "warehouse_name": "Paris Warehouse",
        "city": "Paris",
        "country": "France",
        "region": "Europe",
        "capacity_sqm": 300,
        "timezone": "Europe/Paris",
    },
])

print(f"✅ dim_product: {len(dim_product)} rows")
print(dim_product[["product_name", "category", "base_cost", "base_price"]].head(5).to_string(index=False))

print(f"\n✅ dim_warehouse: {len(dim_warehouse)} rows")
print(dim_warehouse[["warehouse_name", "city", "country", "region"]].to_string(index=False))

StatementMeta(, 1cd907cf-3722-4500-b222-5e782a969438, 6, Finished, Available, Finished, False)

✅ dim_product: 30 rows
       product_name    category  base_cost  base_price
    LED Monitor 24" Electronics        152         211
Mechanical Keyboard Electronics         70          83
       Gaming Mouse Electronics        137         171
  Bluetooth Headset Electronics        393         550
          HD Webcam Electronics        326         435

✅ dim_warehouse: 3 rows
 warehouse_name        city       country        region
  NYC Warehouse    New York United States North America
   LA Warehouse Los Angeles United States North America
Paris Warehouse       Paris        France        Europe


In [5]:
# ============================================================
# CELL 3 (UPDATED): Stock-aware Transaction Generator
# Fix: SALE qty cannot exceed current stock balance
# ============================================================

def generate_transactions(dim_product, dim_warehouse, config):
    start = pd.to_datetime(config["start_date"])
    end = pd.to_datetime(config["end_date"])
    dates = pd.date_range(start, end, freq="D")

    # Seasonality weights
    def seasonal_weight(date):
        m = date.month
        if m in [10, 11, 12]:
            return 2.5
        if m in [6, 7, 8]:
            return 1.5
        return 1.0

    slow_movers = np.random.choice(
        dim_product["product_id"],
        size=6,
        replace=False
    )

    transactions = []
    tx_id = 1
    lot_counter = {}

    # FIX: Running stock tracker {(product_id, warehouse_id): qty}
    stock_tracker = {}

    # Step 1: Initial PURCHASE (opening stock)
    for _, prod in dim_product.iterrows():
        for _, wh in dim_warehouse.iterrows():
            init_qty = np.random.randint(20, 80)
            key = (prod["product_id"], wh["warehouse_id"])
            lot = f"LOT-{prod['product_id']:03d}-001"

            transactions.append({
                "transaction_id": tx_id,
                "date": start,
                "product_id": prod["product_id"],
                "warehouse_id": wh["warehouse_id"],
                "transaction_type": "PURCHASE",
                "quantity": init_qty,
                "unit_cost": prod["base_cost"],
                "unit_price": prod["base_price"],
                "lot_number": lot,
                "reference": "INIT-STOCK",
            })

            stock_tracker[key] = init_qty
            lot_counter[prod["product_id"]] = 1
            tx_id += 1

    # Step 2: Generate daily transactions
    target_tx = config["n_transactions"]
    opening_rows = len(dim_product) * len(dim_warehouse)
    target_total = target_tx + opening_rows

    attempts = 0
    max_attempts = target_tx * 10  # safety limit on loop

    while len(transactions) < target_total:
        attempts += 1
        if attempts > max_attempts:
            break

        date = pd.Timestamp(np.random.choice(dates))
        prod = dim_product.sample(1).iloc[0]
        wh = dim_warehouse.sample(1).iloc[0]
        key = (prod["product_id"], wh["warehouse_id"])

        w = seasonal_weight(date)

        if prod["product_id"] in slow_movers:
            if random.random() > 0.15:
                continue

        tx_type = np.random.choice(
            ["PURCHASE", "SALE", "ADJUSTMENT"],
            p=[0.35, 0.60, 0.05]
        )

        cost_var = prod["base_cost"] * np.random.uniform(0.95, 1.05)
        qty = max(1, int(np.random.randint(1, 15) * w))

        if tx_type == "PURCHASE":
            lot_counter[prod["product_id"]] = lot_counter.get(prod["product_id"], 0) + 1
            lot = f"LOT-{prod['product_id']:03d}-{lot_counter[prod['product_id']]:03d}"
            ref = f"PO-{tx_id:05d}"

            stock_tracker[key] = stock_tracker.get(key, 0) + qty

        elif tx_type == "SALE":
            current = stock_tracker.get(key, 0)

            # Skip sale if no stock available
            if current <= 0:
                continue

            # Cap qty to available stock
            qty = min(qty, current)
            stock_tracker[key] = current - qty

            lot = f"LOT-{prod['product_id']:03d}-001"
            ref = f"SO-{tx_id:05d}"

        else:
            # ADJUSTMENT
            current = stock_tracker.get(key, 0)
            adj = np.random.randint(-3, 4)

            # Adjustment cannot make stock go negative
            adj = max(adj, -current)

            # Avoid zero adjustment to reduce useless rows
            if adj == 0:
                continue

            stock_tracker[key] = current + adj

            qty = adj
            lot = "ADJ"
            ref = f"ADJ-{tx_id:05d}"

        transactions.append({
            "transaction_id": tx_id,
            "date": date,
            "product_id": prod["product_id"],
            "warehouse_id": wh["warehouse_id"],
            "transaction_type": tx_type,
            "quantity": qty,
            "unit_cost": round(cost_var, 0),
            "unit_price": prod["base_price"],
            "lot_number": lot,
            "reference": ref,
        })

        tx_id += 1

    df = pd.DataFrame(transactions)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["date", "transaction_id"]).reset_index(drop=True)

    # Verify: confirm no negative stock
    min_stock = min(stock_tracker.values()) if stock_tracker else 0
    print(f"✅ Stock validation: min balance = {min_stock} (must be ≥ 0)")
    assert min_stock >= 0, "❌ Negative stock detected — check logic!"

    return df


fact_transactions = generate_transactions(
    dim_product,
    dim_warehouse,
    CONFIG
)

print(f"✅ fact_transactions: {len(fact_transactions)} rows")

print("\n📊 Transaction breakdown:")
print(fact_transactions["transaction_type"].value_counts())

print(f"\n📦 Unique products with stock data: {fact_transactions['product_id'].nunique()}")

StatementMeta(, 1cd907cf-3722-4500-b222-5e782a969438, 7, Finished, Available, Finished, False)

✅ Stock validation: min balance = 0 (must be ≥ 0)
✅ fact_transactions: 890 rows

📊 Transaction breakdown:
transaction_type
SALE          455
PURCHASE      396
ADJUSTMENT     39
Name: count, dtype: int64

📦 Unique products with stock data: 30


In [7]:
# ============================================================
# CELL 4: Save Tables to Lakehouse
# ============================================================

def save_to_lakehouse(df_pandas, table_name, mode="overwrite"):
    """Convert pandas → Spark → save as Delta table"""
    spark_df = spark.createDataFrame(df_pandas)

    (
        spark_df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    count = spark.table(table_name).count()
    print(f"✅ {table_name}: {count} rows saved")


print("💾 Saving to Lakehouse...")
print("-" * 45)

# Save tất cả 3 tables
tables = {
    "fact_transactions": fact_transactions,
    "dim_product": dim_product,
    "dim_warehouse": dim_warehouse,
}

for name, df in tables.items():
    save_to_lakehouse(df, name)

# ── Tạo thêm dim_date ───────────────────────────────────────
print("⏳ Generating dim_date...")

dates = pd.date_range("2025-01-01", pd.Timestamp.today().normalize())

dim_date = pd.DataFrame({
    "date": dates,
    "year": dates.year,
    "quarter": dates.quarter,
    "month": dates.month,
    "month_name": dates.strftime("%b"),
    "week": dates.isocalendar().week.astype("int"),
    "day_of_week": dates.day_name(),
    "is_weekend": (dates.dayofweek >= 5).astype("int"),
})

save_to_lakehouse(dim_date, "dim_date")

print("-" * 45)
print("🎉 Done! 4 Delta tables created in Lakehouse.")
print("👉 Check Tables panel (left sidebar) để verify.")

StatementMeta(, 1cd907cf-3722-4500-b222-5e782a969438, 9, Finished, Available, Finished, False)

💾 Saving to Lakehouse...
---------------------------------------------
✅ fact_transactions: 890 rows saved
✅ dim_product: 30 rows saved
✅ dim_warehouse: 3 rows saved
⏳ Generating dim_date...
✅ dim_date: 435 rows saved
---------------------------------------------
🎉 Done! 4 Delta tables created in Lakehouse.
👉 Check Tables panel (left sidebar) để verify.
